In [1]:
!pip install timm --quiet

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import csv
import time
from pathlib import Path

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [5]:
from torch.utils.data import DataLoader, Dataset

In [6]:
!rsync -ah --info=progress2 \
    "/content/drive/MyDrive/Aegis_Safe_Work/tensors/" \
    "/content/tensors/"

         10.04G 100%    5.54MB/s    0:28:48 (xfr#2085, to-chk=0/2093)


In [7]:
import time, numpy as np
from pathlib import Path
files = list(Path("/content/tensors/train").rglob("*.npy"))[:500]
t0 = time.time()
for f in files:
    np.load(f)
print(f"{500/(time.time()-t0):.1f} tensors/sec")

683.0 tensors/sec


In [8]:

class Config:
    # Paths
    TENSORS_ROOT   = Path("/content/tensors")
    MANIFEST_CSV   = Path("/content/drive/MyDrive/Aegis_Safe_Work/manifests/manifest.csv")
    OUTPUT_DIR     = Path("/content/drive/MyDrive/Aegis_Safe_Work/models")
    BEST_MODEL     = OUTPUT_DIR / "fall_detector_best.pt"
    FINAL_MODEL    = OUTPUT_DIR / "fall_detector_final.pt"
    CURVES_PNG     = OUTPUT_DIR / "training_curves.png"
    CONFMAT_PNG    = OUTPUT_DIR / "confusion_matrix_val.png"

    # Tensor shape
    N_FRAMES       = 16
    IMG_SIZE       = 224

    # EfficientNet-Lite0 feature dim (after global avg pool)
    EFFICIENTNET_DIM = 1280

    # Attention MLP hidden dims (same as Aegis-Traffic)
    ATTN_HIDDEN    = 128        # attention scorer hidden

    # Classifier MLP hidden dims
    CLS_HIDDEN_1   = 512
    CLS_HIDDEN_2   = 128
    DROPOUT        = 0.2

    # Training
    BATCH_SIZE     = 32
    NUM_WORKERS    = 4
    PIN_MEMORY     = True

    # Class weights: ratio 1.31:1 normal/fall
    CLASS_WEIGHTS  = {0: 1.0, 1: 1.31}

    # Fall threshold for evaluation
    FALL_THRESHOLD = 0.65

    # Stage 1: frozen backbone — MLP + Attention only
    STAGE1_EPOCHS  = 5
    LR_MLP_S1      = 3e-3

    # Stage 2: unfreeze blocks[3,4,5] + conv_head
    STAGE2_EPOCHS  = 20
    LR_MLP_S2      = 8e-4
    LR_CNN_S2      = 5e-5

    WEIGHT_DECAY   = 2e-4

    # Early stopping
    ES_PATIENCE    = 5
    ES_MIN_DELTA   = 1e-4

    # Seed
    SEED           = 42


In [9]:
cfg = Config()

In [10]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
print("=" * 60)
print("AegisSafeRoad — Training Pipeline")
print("=" * 60)
print(f"Device          : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Tensors dir     : {cfg.TENSORS_ROOT}")
print(f"Training output : {cfg.OUTPUT_DIR}")
print(f"Stage 1 epochs  : {cfg.STAGE1_EPOCHS}  (MLP only)")
print(f"Stage 2 epochs  : {cfg.STAGE2_EPOCHS}  (MLP + CNN fine-tune)")
print(f"Learning rate MLP (Stage 1) : {cfg.LR_MLP_S1}")
print(f"Manifest.csv path  {cfg.MANIFEST_CSV}")
print(f"Batch size      : {cfg.BATCH_SIZE}")
print(f"Early stopping  : patience={cfg.ES_PATIENCE}")
print("=" * 60)

AegisSafeRoad — Training Pipeline
Device          : cuda
GPU             : NVIDIA L4
VRAM            : 23.7 GB
Tensors dir     : /content/tensors
Training output : /content/drive/MyDrive/Aegis_Safe_Work/models
Stage 1 epochs  : 5  (MLP only)
Stage 2 epochs  : 20  (MLP + CNN fine-tune)
Learning rate MLP (Stage 1) : 0.003
Manifest.csv path  /content/drive/MyDrive/Aegis_Safe_Work/manifests/manifest.csv
Batch size      : 32
Early stopping  : patience=5


In [13]:
# FALL && NORMAL DATASET
class FallDataset(Dataset):
    """
    Reads manifest.csv, loads .npy tensors (16,3,224,224) float16.
    Casts to float32 for model forward pass.
    """

    def __init__(self, manifest_csv: Path, split: str, tensors_root: Path):
        self.samples = []
        with open(manifest_csv, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    # Remap path from Drive to /content/tensors/
                    original = Path(row["tensor_path"])
                    # original: .../Aegis_Safe_Work/tensors/train/fall/Fall_XXXX.npy
                    # remap to: /content/tensors/train/fall/Fall_XXXX.npy
                    parts = original.parts
                    tensors_idx = None
                    for i, p in enumerate(parts):
                        if p == "tensors":
                            tensors_idx = i
                            break
                    if tensors_idx is not None:
                        rel = Path(*parts[tensors_idx + 1:])
                        remapped = tensors_root / rel
                    else:
                        remapped = original
                    self.samples.append((remapped, int(row["label"])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        tensor = np.load(str(path))                      # (16, 3, 224, 224) float16
        tensor = torch.from_numpy(tensor.astype(np.float32))  # float32
        return tensor, label


In [15]:
# TEMPORAL ATTENTION
class TemporalAttention(nn.Module):
    """
    Input  : (B, N_FRAMES, EFFICIENTNET_DIM)
    Output : (B, EFFICIENTNET_DIM)  weighted sum over frames
    """

    def __init__(self, input_dim: int = Config.EFFICIENTNET_DIM):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, Config.ATTN_HIDDEN),
            nn.Tanh(),
            nn.Linear(Config.ATTN_HIDDEN, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scores  = self.attention(x)               # (B, 16, 1)
        weights = torch.softmax(scores, dim=1)    # (B, 16, 1)
        pooled  = (weights * x).sum(dim=1)        # (B, 1280)
        return pooled


In [41]:

class FallDetector(nn.Module):
    """
    EfficientNet-Lite0 + Temporal Attention + MLP classifier.

    Forward:
        (B, 16, 3, 224, 224)
        -> per-frame EfficientNet-Lite0  -> (B, 16, 1280)
        -> TemporalAttention             -> (B, 1280)
        -> MLP classifier                -> (B, 1)
        -> sigmoid                       -> probability [0, 1]
    """

    def __init__(self):
        super().__init__()

        # EfficientNet-Lite0 backbone — ImageNet pretrained
        backbone = timm.create_model("efficientnet_lite0", pretrained=True)

        # Actual timm EfficientNet-Lite0 structure (verified):
        #   backbone.conv_stem   -> Conv2d(3, 32, 3x3)
        #   backbone.bn1         -> BatchNormAct2d (includes ReLU6 internally)
        #   backbone.blocks[0-6] -> 7 inverted residual blocks
        #   backbone.conv_head   -> Conv2d(320, 1280, 1x1)
        #   backbone.bn2         -> BatchNormAct2d (includes ReLU6 internally)
        #   backbone.global_pool -> SelectAdaptivePool2d (avg + flatten)
        #   backbone.classifier  -> Linear(1280, 1000)  <- removed
        self.conv_stem   = backbone.conv_stem
        self.bn1         = backbone.bn1         # BatchNormAct2d: BN + ReLU6
        self.blocks      = backbone.blocks      # nn.Sequential of 7 blocks (0-6)
        self.conv_head   = backbone.conv_head
        self.bn2         = backbone.bn2         # BatchNormAct2d: BN + ReLU6
        self.global_pool = backbone.global_pool # SelectAdaptivePool2d: avg + flatten -> (B, 1280)

        # Temporal Attention
        self.temporal_attention = TemporalAttention(Config.EFFICIENTNET_DIM)

        # MLP Classifier
        self.classifier = nn.Sequential(
            nn.Linear(Config.EFFICIENTNET_DIM, Config.CLS_HIDDEN_1),
            nn.ReLU(inplace=True),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(Config.CLS_HIDDEN_1, Config.CLS_HIDDEN_2),
            nn.ReLU(inplace=True),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(Config.CLS_HIDDEN_2, 1),
            # No Sigmoid here — BCEWithLogitsLoss applies it internally
            # Use torch.sigmoid(logits) explicitly at inference/evaluation
        )

        # Freeze entire backbone initially (Stage 1)
        # Called AFTER temporal_attention and classifier are assigned
        self._freeze_all()

    # ------------------------------------------------------------------
    # Freeze / Unfreeze helpers
    # ------------------------------------------------------------------

    def _freeze_all(self):
        for param in self.parameters():
            param.requires_grad = False
        # MLP + Attention always trainable
        for param in self.temporal_attention.parameters():
            param.requires_grad = True
        for param in self.classifier.parameters():
            param.requires_grad = True

    def freeze_backbone(self):
        """Stage 1: freeze entire EfficientNet-Lite0."""
        for module in [self.conv_stem, self.bn1, self.blocks, self.conv_head, self.bn2]:
            for param in module.parameters():
                param.requires_grad = False
        print("[Model] EfficientNet-Lite0 fully frozen")

    def unfreeze_stage2(self):
        """
        Stage 2: unfreeze blocks[4], blocks[5], blocks[6] + conv_head + bn2.
        blocks[0-3] remain frozen.
        7 total blocks (0-6) verified from timm efficientnet_lite0 architecture.
        """
        # Ensure everything frozen first
        self.freeze_backbone()

        # Unfreeze last 3 blocks
        for i in [4, 5, 6]:
            for param in self.blocks[i].parameters():
                param.requires_grad = True

        # Unfreeze conv_head + bn2
        for param in self.conv_head.parameters():
            param.requires_grad = True
        for param in self.bn2.parameters():
            param.requires_grad = True

        print("[Model] Unfrozen: blocks[4,5,6] + conv_head + bn2")
        self._print_trainable()

    def _print_trainable(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"[Model] Trainable: {trainable:,} / {total:,} params")

    def get_param_groups(self, lr_mlp: float, lr_cnn: float) -> list:
        """Differential LR: CNN params -> lr_cnn, MLP+Attention -> lr_mlp."""
        cnn_params = []
        for i in [4, 5, 6]:
            cnn_params += list(self.blocks[i].parameters())
        cnn_params += list(self.conv_head.parameters())
        cnn_params += list(self.bn2.parameters())
        cnn_params  = [p for p in cnn_params if p.requires_grad]

        mlp_params = (
            list(self.temporal_attention.parameters()) +
            list(self.classifier.parameters())
        )
        return [
            {"params": cnn_params, "lr": lr_cnn},
            {"params": mlp_params, "lr": lr_mlp},
        ]

    def count_parameters(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable

    # ------------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------------

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 16, 3, 224, 224) float32
        Returns:
            (B, 1) sigmoid probability
        """
        B, T, C, H, W = x.shape

        # Process all frames in one batch through EfficientNet-Lite0
        x_flat = x.view(B * T, C, H, W)              # (B*16, 3, 224, 224)

        # Forward through backbone — bn1/bn2 are BatchNormAct2d (include activation)
        f = self.conv_stem(x_flat)                    # (B*16, 32, 112, 112)
        f = self.bn1(f)                               # (B*16, 32, 112, 112) + ReLU6
        f = self.blocks(f)                            # (B*16, 320, 7, 7)
        f = self.conv_head(f)                         # (B*16, 1280, 7, 7)
        f = self.bn2(f)                               # (B*16, 1280, 7, 7) + ReLU6
        f = self.global_pool(f)                       # (B*16, 1280) — avg pool + flatten
        f = f.view(B, T, Config.EFFICIENTNET_DIM)     # (B, 16, 1280)

        # Temporal attention pooling
        pooled = self.temporal_attention(f)           # (B, 1280)

        # MLP classification
        out = self.classifier(pooled)                 # (B, 1)
        return out


In [43]:
import timm
backbone = timm.create_model("efficientnet_lite0", pretrained=True)
print(backbone)

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): ReLU6(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU6(inplace=True)
        )
        (aa): Identity()
        (se): Identity()
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): Identity()
        )
        (drop_path): Identity()
      )
    )
    (1): Sequent

In [51]:
import timm
import torch.nn as nn

# Instantiate the EfficientNet-Lite0 model
backbone = timm.create_model("efficientnet_lite0", pretrained=True)

print("\n--- EfficientNet-Lite0 Model Structure ---")
print(backbone)

print("\n--- Available Attributes (partial list) ---")
# You can also use dir(backbone) to see all attributes and methods
# Here are some of the key attributes you might be interested in:
print(f"conv_stem: {backbone.conv_stem.__class__.__name__}")
print(f"bn1: {backbone.bn1.__class__.__name__}")
print(f"blocks: {backbone.blocks.__class__.__name__} (contains {len(backbone.blocks)} blocks)")
print(f"conv_head: {backbone.conv_head.__class__.__name__}")
print(f"bn2: {backbone.bn2.__class__.__name__}")
print(f"global_pool: {backbone.global_pool.__class__.__name__}")
print(f"classifier: {backbone.classifier.__class__.__name__}")


--- EfficientNet-Lite0 Model Structure ---
EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): ReLU6(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU6(inplace=True)
        )
        (aa): Identity()
        (se): Identity()
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): Identity()
        )
        (drop_path

In [44]:
# EARLY STOPPING

class EarlyStopping:
    def __init__(
        self,
        patience: int  = Config.ES_PATIENCE,
        min_delta: float = Config.ES_MIN_DELTA,
        save_path: Path  = Config.BEST_MODEL,
    ):
        self.patience  = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.best_loss = float("inf")
        self.counter   = 0
        self.best_epoch = 0
        self.stop      = False

    def step(self, val_loss: float, model: nn.Module, epoch: int):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_epoch = epoch
            torch.save(model.state_dict(), str(self.save_path))
            print(
                f"    [EarlyStopping] Best model saved  "
                f"val_loss={val_loss:.4f}  epoch={epoch}"
            )
        else:
            self.counter += 1
            print(
                f"    [EarlyStopping] No improvement since:  "
                f"counter={self.counter}/{self.patience}  "
                f"best={self.best_loss:.4f} @ epoch {self.best_epoch}"
            )
            if self.counter >= self.patience:
                self.stop = True
                print(
                    f"    [EarlyStopping] Triggered at epoch {epoch}. "
                    f"Best epoch was {self.best_epoch}."
                )


In [45]:
# TRAINING FUNCTION

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epoch: int,
) -> dict:
    model.train()
    total_loss = 0.0
    all_preds  = []
    all_labels = []
    t0         = time.time()

    for batch_idx, (x, labels) in enumerate(loader):
        x      = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(x)                     # (B, 1)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        probs = torch.sigmoid(logits.detach().cpu().squeeze(1))
        preds = (probs >= Config.FALL_THRESHOLD).long()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.detach().cpu().squeeze(1).long().tolist())

    n        = len(loader.dataset)
    avg_loss = total_loss / n
    acc      = sum(p == l for p, l in zip(all_preds, all_labels)) / n
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    prec     = precision_score(all_labels, all_preds, zero_division=0)
    rec      = recall_score(all_labels, all_preds, zero_division=0)
    elapsed  = time.time() - t0

    print(
        f"  [Train] loss={avg_loss:.4f}  acc={acc:.4f}  "
        f"f1={f1:.4f}  prec={prec:.4f}  rec={rec:.4f}  "
        f"t={elapsed:.1f}s"
    )
    return {"loss": avg_loss, "acc": acc, "f1": f1, "prec": prec, "rec": rec}


In [46]:
# EVALUATE FUNCTION

@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for x, labels in loader:
        x      = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().unsqueeze(1)
        logits = model(x)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * x.size(0)
        probs = torch.sigmoid(logits.cpu().squeeze(1))
        preds = (probs >= Config.FALL_THRESHOLD).long()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().squeeze(1).long().tolist())

    n        = len(loader.dataset)
    avg_loss = total_loss / n
    acc      = sum(p == l for p, l in zip(all_preds, all_labels)) / n
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    prec     = precision_score(all_labels, all_preds, zero_division=0)
    rec      = recall_score(all_labels, all_preds, zero_division=0)

    print(
        f"  [Val]   loss={avg_loss:.4f}  acc={acc:.4f}  "
        f"f1={f1:.4f}  prec={prec:.4f}  rec={rec:.4f}"
    )
    return {
        "loss": avg_loss, "acc": acc, "f1": f1,
        "prec": prec, "rec": rec,
        "preds": all_preds, "labels": all_labels,
    }


In [47]:
# PLOTTING TRAINING CURVES

def plot_curves(history: dict, save_path: Path):
    """Plot loss, accuracy, f1, precision, recall, and LR over epochs."""
    epochs = range(1, len(history["train_loss"]) + 1)
    stage2_start = Config.STAGE1_EPOCHS + 0.5

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Aegis-Safe-Work | FallDetector Training Curves", fontsize=14)

    metrics = [
        ("loss",  "Loss",      axes[0, 0]),
        ("acc",   "Accuracy",  axes[0, 1]),
        ("f1",    "F1 Score",  axes[0, 2]),
        ("prec",  "Precision", axes[1, 0]),
        ("rec",   "Recall",    axes[1, 1]),
    ]

    for key, title, ax in metrics:
        ax.plot(epochs, history[f"train_{key}"], label="Train", marker="o", markersize=3)
        ax.plot(epochs, history[f"val_{key}"],   label="Val",   marker="s", markersize=3)
        ax.axvline(x=stage2_start, color="red", linestyle="--", alpha=0.6, label="Stage 2 start")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)

    # LR plot
    ax_lr = axes[1, 2]
    ax_lr.plot(epochs, history["lr_mlp"], label="LR MLP",  marker="o", markersize=3)
    if any(lr > 0 for lr in history["lr_cnn"]):
        ax_lr.plot(epochs, history["lr_cnn"], label="LR CNN", marker="s", markersize=3)
    ax_lr.axvline(x=stage2_start, color="red", linestyle="--", alpha=0.6, label="Stage 2 start")
    ax_lr.set_title("Learning Rate")
    ax_lr.set_xlabel("Epoch")
    ax_lr.set_yscale("log")
    ax_lr.legend()
    ax_lr.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Training curves saved: {save_path}")


def plot_confusion_matrix(labels: list, preds: list, save_path: Path):
    cm   = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fall"])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(
        f"Aegis-Safe-Work | Val Confusion Matrix\n"
        f"threshold={Config.FALL_THRESHOLD}"
    )
    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Confusion matrix saved: {save_path}")




In [48]:
def print_trainiable_parameters(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Model] Total parameters: {total:,}")
    print(f"[Model] Trainable parameters: {trainable:,}")

In [49]:
#  MAIN TRAINING RUN FUNCTION TWO STAGES

def training_run():
    torch.manual_seed(Config.SEED)
    np.random.seed(Config.SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")
    if device.type == "cuda":
        print(f"[GPU] {torch.cuda.get_device_name(0)}")

    Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    # Datasets + Loaders

    print("\n[Data] Loading manifest...")
    train_ds = FallDataset(Config.MANIFEST_CSV, "train", Config.TENSORS_ROOT)
    val_ds   = FallDataset(Config.MANIFEST_CSV, "val",   Config.TENSORS_ROOT)
    print(f"[Data] Train: {len(train_ds)} | Val: {len(val_ds)}")

    train_loader = DataLoader(
        train_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.PIN_MEMORY,
    )

    # Model

    print("\n[Model] Building FallDetector...")
    model = FallDetector().to(device)
    print_trainiable_parameters(model)

    # Loss with class weights

    pos_weight = torch.tensor(
        [Config.CLASS_WEIGHTS[1] / Config.CLASS_WEIGHTS[0]],
        dtype=torch.float32,
        device=device,
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


    # History

    history = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "train_prec": [], "train_rec": [],
        "val_loss":   [], "val_acc":   [], "val_f1":   [],
        "val_prec":   [], "val_rec":   [],
        "lr_mlp": [], "lr_cnn": [],
    }

    early_stopping = EarlyStopping()
    total_epochs   = Config.STAGE1_EPOCHS + Config.STAGE2_EPOCHS


    # STAGE 1: frozen backbone — MLP + Attention only

    print("\n" + "=" * 60)
    print(f"STAGE 1 — Epochs 1-{Config.STAGE1_EPOCHS} | Backbone frozen")
    print("=" * 60)

    model.freeze_backbone()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=Config.LR_MLP_S1,
        weight_decay=Config.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.STAGE1_EPOCHS, eta_min=1e-5
    )

    for epoch in range(1, Config.STAGE1_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{total_epochs}  [Stage 1]")
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        vl = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        for k in ["loss", "acc", "f1", "prec", "rec"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])

        current_lr = optimizer.param_groups[0]["lr"]
        history["lr_mlp"].append(current_lr)
        history["lr_cnn"].append(0.0)

        early_stopping.step(vl["loss"], model, epoch)
        if early_stopping.stop:
            print("[Stage 1] Early stopping triggered.")
            break


    # STAGE 2: unfreeze blocks[3,4,5] + conv_head

    print("\n" + "=" * 60)
    print(f"STAGE 2 — Epochs {Config.STAGE1_EPOCHS+1}-{total_epochs} | Partial unfreeze")
    print("=" * 60)

    model.unfreeze_stage2()
    param_groups = model.get_param_groups(
        lr_mlp=Config.LR_MLP_S2,
        lr_cnn=Config.LR_CNN_S2,
    )
    optimizer = torch.optim.AdamW(
        param_groups,
        weight_decay=Config.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.STAGE2_EPOCHS, eta_min=1e-6
    )

    early_stopping = EarlyStopping()   # reset counter for Stage 2

    for epoch in range(Config.STAGE1_EPOCHS + 1, total_epochs + 1):
        print(f"\nEpoch {epoch}/{total_epochs}  [Stage 2]")
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        vl = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        for k in ["loss", "acc", "f1", "prec", "rec"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])

        lr_mlp = optimizer.param_groups[1]["lr"]
        lr_cnn = optimizer.param_groups[0]["lr"]
        history["lr_mlp"].append(lr_mlp)
        history["lr_cnn"].append(lr_cnn)

        early_stopping.step(vl["loss"], model, epoch)
        if early_stopping.stop:
            print("[Stage 2] Early stopping triggered.")
            break


    # SAVE FINAL MODEL + PLOTS

    torch.save(model.state_dict(), str(Config.FINAL_MODEL))
    print(f"\n[Model] Final weights saved: {Config.FINAL_MODEL}")

    # Load best model for confusion matrix
    model.load_state_dict(torch.load(str(Config.BEST_MODEL), map_location=device))
    print(f"[Model] Best weights loaded from: {Config.BEST_MODEL}")

    final_val = evaluate(model, val_loader, criterion, device)

    plot_curves(history, Config.CURVES_PNG)
    plot_confusion_matrix(final_val["labels"], final_val["preds"], Config.CONFMAT_PNG)


    # Final summary

    print("\n" + "=" * 60)
    print("TRAINING COMPLETE — FINAL VAL METRICS (best model)")
    print("=" * 60)
    print(f"  Loss      : {final_val['loss']:.4f}")
    print(f"  Accuracy  : {final_val['acc']:.4f}")
    print(f"  F1        : {final_val['f1']:.4f}")
    print(f"  Precision : {final_val['prec']:.4f}")
    print(f"  Recall    : {final_val['rec']:.4f}")
    print(f"  Threshold : {Config.FALL_THRESHOLD}")
    print(f"  Best epoch: {early_stopping.best_epoch}")
    print("=" * 60)

In [50]:
training_run()

[Device] cuda
[GPU] NVIDIA L4

[Data] Loading manifest...
[Data] Train: 1875 | Val: 210

[Model] Building FallDetector...
[Model] Total parameters: 4,256,770
[Model] Trainable parameters: 885,762

STAGE 1 — Epochs 1-5 | Backbone frozen
[Model] EfficientNet-Lite0 fully frozen

Epoch 1/25  [Stage 1]
  [Train] loss=0.3840  acc=0.8469  f1=0.8152  prec=0.8531  rec=0.7805  t=30.9s
  [Val]   loss=0.1910  acc=0.9476  f1=0.9418  prec=0.9082  rec=0.9780
    [EarlyStopping] Best model saved  val_loss=0.1910  epoch=1

Epoch 2/25  [Stage 1]
  [Train] loss=0.1851  acc=0.9280  f1=0.9164  prec=0.9204  rec=0.9125  t=26.9s
  [Val]   loss=0.1392  acc=0.9429  f1=0.9318  prec=0.9647  rec=0.9011
    [EarlyStopping] Best model saved  val_loss=0.1392  epoch=2

Epoch 3/25  [Stage 1]
  [Train] loss=0.1365  acc=0.9461  f1=0.9365  prec=0.9551  rec=0.9186  t=27.1s
  [Val]   loss=0.1550  acc=0.9429  f1=0.9375  prec=0.8911  rec=0.9890
    [EarlyStopping] No improvement since:  counter=1/5  best=0.1392 @ epoch 2

Epo

In [74]:
import cv2
cap = cv2.VideoCapture("/content/videos2test/fall_001.mp4")
print("Width :", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("Height:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("FPS   :", cap.get(cv2.CAP_PROP_FPS))
print("Frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

Width : 1080.0
Height: 1920.0
FPS   : 29.97002997002997
Frames: 399.0


In [82]:
import os
os.makedirs("/content/videos2test", exist_ok=True)

In [83]:
import cv2
import numpy as np
import timm
import torch
import torch.nn as nn


In [84]:
# Config Settings
BEST_MODEL_PATH  = Path("/content/drive/MyDrive/Aegis_Safe_Work/models/fall_detector_best.pt")
VIDEOS_DIR       = Path("/content/videos2test")
N_FRAMES         = 16
IMG_SIZE         = 224
FALL_THRESHOLD   = 0.65
EFFICIENTNET_DIM = 1280

In [85]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


In [88]:

class TemporalAttention(nn.Module):
    def __init__(self, input_dim: int = EFFICIENTNET_DIM):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor):
        scores  = self.attention(x)                  # (B, 16, 1)
        weights = torch.softmax(scores, dim=1)       # (B, 16, 1)
        pooled  = (weights * x).sum(dim=1)           # (B, 1280)
        return pooled, weights.squeeze(-1)           # also return weights


In [89]:

class FallDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = timm.create_model("efficientnet_lite0", pretrained=False)
        self.conv_stem   = backbone.conv_stem
        self.bn1         = backbone.bn1
        self.blocks      = backbone.blocks
        self.conv_head   = backbone.conv_head
        self.bn2         = backbone.bn2
        self.global_pool = backbone.global_pool

        self.temporal_attention = TemporalAttention(EFFICIENTNET_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(EFFICIENTNET_DIM, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor):
        B, T, C, H, W = x.shape
        x_flat = x.view(B * T, C, H, W)
        f = self.conv_stem(x_flat)
        f = self.bn1(f)
        f = self.blocks(f)
        f = self.conv_head(f)
        f = self.bn2(f)
        f = self.global_pool(f)                      # (B*16, 1280)
        f = f.view(B, T, EFFICIENTNET_DIM)           # (B, 16, 1280)

        pooled, attn_weights = self.temporal_attention(f)  # (B,1280), (B,16)
        logit = self.classifier(pooled)              # (B, 1)
        return logit, attn_weights



In [90]:

def letterbox_frame(frame_bgr: np.ndarray, target: int = IMG_SIZE) -> np.ndarray:
    """
    Resize frame to target x target with black padding (letterbox).
    Matches ETL Stage 1 ffmpeg filter:
        scale=224:224:force_original_aspect_ratio=decrease,
        pad=224:224:(ow-iw)/2:(oh-ih)/2:color=black
    Returns RGB frame (target, target, 3) uint8.
    """
    h, w   = frame_bgr.shape[:2]
    scale  = target / max(h, w)
    new_w  = int(w * scale)
    new_h  = int(h * scale)

    resized = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # Black canvas
    canvas  = np.zeros((target, target, 3), dtype=np.uint8)

    # Center paste
    pad_top  = (target - new_h) // 2
    pad_left = (target - new_w) // 2
    canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized

    # BGR -> RGB
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)



In [91]:

def video_to_tensor(video_path: Path):
    """
    Open video, apply letterbox 224x224, sample N_FRAMES uniformly via linspace.
    Preprocessing matches ETL Stage 1 exactly.
    Returns:
        tensor       : (1, 16, 3, 224, 224) float32 — ready for model
        total_frames : int
        fps_original : float
        duration_s   : float
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_original = cap.get(cv2.CAP_PROP_FPS)
    duration_s   = total_frames / fps_original if fps_original > 0 else 0.0

    indices = np.linspace(0, total_frames - 1, N_FRAMES, dtype=int)
    indices = np.clip(indices, 0, total_frames - 1)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            fallback = frames[-1].copy() if frames else np.zeros(
                (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8
            )
            frames.append(fallback)
            continue
        frames.append(letterbox_frame(frame, IMG_SIZE))   # letterbox + BGR->RGB

    cap.release()

    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0  # (16, 224, 224, 3)
    arr = (arr - MEAN) / STD                                    # ImageNet norm
    arr = arr.transpose(0, 3, 1, 2)                            # (16, 3, 224, 224)
    tensor = torch.from_numpy(arr).unsqueeze(0)                # (1, 16, 3, 224, 224)

    return tensor, total_frames, fps_original, duration_s



In [92]:

def print_attention_log(
    video_name: str,
    attn_weights: np.ndarray,
    total_frames: int,
    fps: float,
    duration_s: float,
    prob: float,
    prediction: str,
    elapsed_ms: float,
):
    """Print per-frame attention weights with frame index and timestamp."""
    indices  = np.linspace(0, total_frames - 1, N_FRAMES, dtype=int)
    bar_max  = 30  # max bar width for visualization

    print(f"\n{'='*65}")
    print(f"Video : {video_name}")
    print(f"Frames: {total_frames} total | FPS: {fps:.1f} | Duration: {duration_s:.2f}s")
    print(f"{'='*65}")
    print(f"  {'Frame':>6}  {'Time':>6}  {'Attn':>6}  {'Bar'}")
    print(f"  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*bar_max}")

    for i, (frame_idx, weight) in enumerate(zip(indices, attn_weights)):
        timestamp = frame_idx / fps if fps > 0 else 0.0
        bar_len   = int(weight * bar_max / attn_weights.max())
        bar       = "#" * bar_len
        # Mark peak attention frame
        peak_mark = " <-- peak" if weight == attn_weights.max() else ""
        print(f"  {frame_idx:>6}  {timestamp:>5.2f}s  {weight:>6.4f}  {bar}{peak_mark}")

    print(f"  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*bar_max}")
    print(f"  Attn sum (should be ~1.0): {attn_weights.sum():.6f}")
    print(f"\n  Probability (fall) : {prob:.4f}")
    print(f"  Threshold          : {FALL_THRESHOLD}")
    print(f"  Prediction         : {prediction}")
    print(f"  Inference time     : {elapsed_ms:.1f} ms")
    print(f"{'='*65}")


In [93]:

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    # Load model
    print(f"[Model] Loading weights from: {BEST_MODEL_PATH}")
    model = FallDetector()
    state = torch.load(str(BEST_MODEL_PATH), map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("[Model] Ready.")

    # Collect videos
    videos = sorted(VIDEOS_DIR.glob("*.mp4"))
    if not videos:
        print(f"[ERROR] No .mp4 files found in {VIDEOS_DIR}")
        return

    print(f"\n[Test] Found {len(videos)} videos in {VIDEOS_DIR}")

    results = []

    with torch.no_grad():
        for video_path in videos:
            try:
                tensor, total_frames, fps, duration_s = video_to_tensor(video_path)
                tensor = tensor.to(device)

                t0 = time.time()
                logit, attn_weights = model(tensor)
                elapsed_ms = (time.time() - t0) * 1000

                prob       = torch.sigmoid(logit).item()
                prediction = "FALL" if prob >= FALL_THRESHOLD else "NORMAL"
                attn_np    = attn_weights.squeeze(0).cpu().numpy()  # (16,)

                print_attention_log(
                    video_name   = video_path.name,
                    attn_weights = attn_np,
                    total_frames = total_frames,
                    fps          = fps,
                    duration_s   = duration_s,
                    prob         = prob,
                    prediction   = prediction,
                    elapsed_ms   = elapsed_ms,
                )

                results.append({
                    "video":      video_path.name,
                    "prob":       prob,
                    "prediction": prediction,
                    "frames":     total_frames,
                    "duration_s": duration_s,
                })

            except Exception as e:
                print(f"[ERROR] {video_path.name}: {e}")

    # Summary table
    print(f"\n{'='*65}")
    print("INFERENCE SUMMARY")
    print(f"{'='*65}")
    print(f"  {'Video':<35}  {'Prob':>6}  {'Prediction'}")
    print(f"  {'-'*35}  {'-'*6}  {'-'*10}")
    for r in results:
        print(f"  {r['video']:<35}  {r['prob']:>6.4f}  {r['prediction']}")
    print(f"{'='*65}")
    n_fall   = sum(1 for r in results if r["prediction"] == "FALL")
    n_normal = sum(1 for r in results if r["prediction"] == "NORMAL")
    print(f"  FALL predictions  : {n_fall}")
    print(f"  NORMAL predictions: {n_normal}")
    print(f"{'='*65}")



In [94]:
main()

[Device] cuda
[Model] Loading weights from: /content/drive/MyDrive/Aegis_Safe_Work/models/fall_detector_best.pt
[Model] Ready.

[Test] Found 9 videos in /content/videos2test

Video : business_man_greet_officers.mp4
Frames: 121 total | FPS: 24.0 | Duration: 5.04s
   Frame    Time    Attn  Bar
  ------  ------  ------  ------------------------------
       0   0.00s  0.0519  ##
       8   0.33s  0.0810  ####
      16   0.67s  0.0191  #
      24   1.00s  0.0112  
      32   1.33s  0.0864  ####
      40   1.67s  0.0061  
      48   2.00s  0.0130  
      56   2.33s  0.0018  
      64   2.67s  0.0232  #
      72   3.00s  0.0219  #
      80   3.33s  0.0209  #
      88   3.67s  0.0266  #
      96   4.00s  0.0063  
     104   4.33s  0.0608  ###
     112   4.67s  0.5253  ############################## <-- peak
     120   5.00s  0.0447  ##
  ------  ------  ------  ------------------------------
  Attn sum (should be ~1.0): 1.000000

  Probability (fall) : 0.0057
  Threshold          : 0.65
  Pre

In [112]:
import torch

In [95]:
!pip install onnxruntime-gpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 5.9 MB/s eta 0:00:00


In [114]:
BEST_MODEL_PATH = Path("/content/drive/MyDrive/Aegis_Safe_Work/models/fall_detector_best.pt")
OUTPUT_DIR      = Path("/content/drive/MyDrive/Aegis_Safe_Work/models")
ONNX_FP32       = OUTPUT_DIR / "fall_detector.onnx"
ONNX_OPT        = OUTPUT_DIR / "fall_detector_optimized.onnx"

In [115]:

OPSET           = 17
N_FRAMES        = 16
IMG_SIZE        = 224
EFFICIENTNET_DIM = 1280


In [116]:

class TemporalAttention(nn.Module):
    def __init__(self, input_dim: int = EFFICIENTNET_DIM):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor):
        scores  = self.attention(x)
        weights = torch.softmax(scores, dim=1)
        pooled  = (weights * x).sum(dim=1)
        return pooled


In [117]:

class FallDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = timm.create_model("efficientnet_lite0", pretrained=False)
        self.conv_stem   = backbone.conv_stem
        self.bn1         = backbone.bn1
        self.blocks      = backbone.blocks
        self.conv_head   = backbone.conv_head
        self.bn2         = backbone.bn2
        self.global_pool = backbone.global_pool

        self.temporal_attention = TemporalAttention(EFFICIENTNET_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(EFFICIENTNET_DIM, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Export-friendly forward: returns sigmoid probability directly.
        Input : (B, 16, 3, 224, 224) float32
        Output: (B, 1) float32 in [0, 1]
        """
        B, T, C, H, W = x.shape
        x_flat = x.view(B * T, C, H, W)
        f = self.conv_stem(x_flat)
        f = self.bn1(f)
        f = self.blocks(f)
        f = self.conv_head(f)
        f = self.bn2(f)
        f = self.global_pool(f)
        f = f.view(B, T, EFFICIENTNET_DIM)
        pooled = self.temporal_attention(f)
        logit  = self.classifier(pooled)
        return torch.sigmoid(logit)        # sigmoid baked in for ONNX Runtime



In [118]:

def export_onnx(model: nn.Module, path: Path, device: torch.device) -> None:
    model.eval()
    dummy = torch.randn(1, N_FRAMES, 3, IMG_SIZE, IMG_SIZE, device=device)

    torch.onnx.export(
    model,
    dummy,
    str(path),
    opset_version      = 18,
    input_names        = ["input"],
    output_names       = ["probability"],
    dynamic_axes       = {
        "input":       {0: "batch_size"},
        "probability": {0: "batch_size"},
    },
    do_constant_folding = True,
    export_params       = True,
    dynamo             = False,    # fuerza el exporter legacy — un solo archivo
)
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"[Export] {path.name}  ->  {size_mb:.2f} MB")


In [119]:

def optimize_onnx(src: Path, dst: Path) -> None:
    import onnxruntime as ort
    from onnxruntime.transformers.optimizer import optimize_model

    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )
    sess_options.optimized_model_filepath = str(dst)

    # Trigger optimization by creating a session — ORT writes optimized graph
    ort.InferenceSession(
        str(src),
        sess_options,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
    )
    size_mb = dst.stat().st_size / 1024 / 1024
    print(f"[Optimize] {dst.name}  ->  {size_mb:.2f} MB")


In [120]:

def validate_onnx(model: nn.Module, onnx_path: Path, device: torch.device) -> None:
    import onnxruntime as ort

    model.eval()
    dummy_np = np.random.randn(1, N_FRAMES, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    dummy_pt = torch.from_numpy(dummy_np).to(device)

    with torch.no_grad():
        pt_out = model(dummy_pt).cpu().numpy()

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    sess      = ort.InferenceSession(str(onnx_path), providers=providers)
    ort_out   = sess.run(["probability"], {"input": dummy_np})[0]

    max_diff  = np.abs(pt_out - ort_out).max()
    print(f"[Validate] PyTorch output : {pt_out.flatten()}")
    print(f"[Validate] ONNX RT output : {ort_out.flatten()}")
    print(f"[Validate] Max abs diff   : {max_diff:.8f}")

    if max_diff < 1e-4:
        print("[Validate] PASS — outputs match within tolerance 1e-4")
    else:
        print("[Validate] WARN — diff exceeds 1e-4, inspect graph")


In [121]:

def benchmark_onnx(onnx_path: Path, n_runs: int = 50) -> None:
    import time
    import onnxruntime as ort

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    sess      = ort.InferenceSession(str(onnx_path), providers=providers)
    dummy     = np.random.randn(1, N_FRAMES, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)

    # Warmup
    for _ in range(5):
        sess.run(["probability"], {"input": dummy})

    t0 = time.time()
    for _ in range(n_runs):
        sess.run(["probability"], {"input": dummy})
    elapsed_ms = (time.time() - t0) / n_runs * 1000

    print(f"[Benchmark] {n_runs} runs | avg latency: {elapsed_ms:.2f} ms/inference")
    print(f"[Benchmark] Throughput  : {1000/elapsed_ms:.1f} inferences/sec")


In [122]:

def make_onnx_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load model
    print(f"\n[Model] Loading weights: {BEST_MODEL_PATH}")
    model = FallDetector()
    state = torch.load(str(BEST_MODEL_PATH), map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("[Model] Ready.")

    # Export fp32
    print(f"\n[Step 1] Exporting ONNX opset {OPSET}...")
    export_onnx(model, ONNX_FP32, device)

    # Graph optimization via ORT
    print("\n[Step 2] Optimizing graph with ORT_ENABLE_ALL...")
    optimize_onnx(ONNX_FP32, ONNX_OPT)

    # Validate optimized model
    print("\n[Step 3] Validating optimized ONNX vs PyTorch...")
    validate_onnx(model, ONNX_OPT, device)

    # Benchmark
    print("\n[Step 4] Benchmarking optimized ONNX...")
    benchmark_onnx(ONNX_OPT)

    # Final summary
    fp32_mb = ONNX_FP32.stat().st_size / 1024 / 1024
    opt_mb  = ONNX_OPT.stat().st_size  / 1024 / 1024
    print(f"\n{'='*55}")
    print("ONNX EXPORT SUMMARY")
    print(f"{'='*55}")
    print(f"  fall_detector.onnx           : {fp32_mb:.2f} MB")
    print(f"  fall_detector_optimized.onnx : {opt_mb:.2f} MB")
    print(f"  Opset                        : {OPSET}")
    print(f"  Input  shape                 : (B, 16, 3, 224, 224) float32")
    print(f"  Output shape                 : (B, 1) float32 in [0,1]")
    print(f"  Sigmoid                      : baked into graph")
    print(f"  Threshold at inference       : 0.65")
    print(f"{'='*55}")


In [108]:
!ls -lh /content/drive/MyDrive/Aegis_Safe_Work/models/

total 34M
-rw------- 1 root root  28K Jun 29 18:08 confusion_matrix_val.png
-rw------- 1 root root  17M Jun 29 18:04 fall_detector_best.pt
-rw------- 1 root root  17M Jun 29 18:08 fall_detector_final.pt
-rw------- 1 root root 315K Jun 29 18:08 training_curves.png


In [109]:
!pip uninstall onnxruntime-gpu -y --quiet


In [110]:
!pip install onnxruntime --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 127.2 MB/s eta 0:00:00


In [123]:
make_onnx_model()

[Device] cuda

[Model] Loading weights: /content/drive/MyDrive/Aegis_Safe_Work/models/fall_detector_best.pt
[Model] Ready.

[Step 1] Exporting ONNX opset 17...


/tmp/ipykernel_3820/3912603757.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


[Export] fall_detector.onnx  ->  16.20 MB

[Step 2] Optimizing graph with ORT_ENABLE_ALL...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


[Optimize] fall_detector_optimized.onnx  ->  16.23 MB

[Step 3] Validating optimized ONNX vs PyTorch...
[Validate] PyTorch output : [3.6049148e-06]
[Validate] ONNX RT output : [3.6358833e-06]
[Validate] Max abs diff   : 0.00000003
[Validate] PASS — outputs match within tolerance 1e-4

[Step 4] Benchmarking optimized ONNX...
[Benchmark] 50 runs | avg latency: 52.06 ms/inference
[Benchmark] Throughput  : 19.2 inferences/sec

ONNX EXPORT SUMMARY
  fall_detector.onnx           : 16.20 MB
  fall_detector_optimized.onnx : 16.23 MB
  Opset                        : 17
  Input  shape                 : (B, 16, 3, 224, 224) float32
  Output shape                 : (B, 1) float32 in [0,1]
  Sigmoid                      : baked into graph
  Threshold at inference       : 0.65
